In [377]:
# 如果要用 0813_change_seed.py 要把這個c ell 加上 tag 'parameters'
SEED = 95
N_VAL_POS, N_VAL_NEG = 9, 9
N_NEG_GROUPS, NEG_PER_GROUP = 10, 86

In [378]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, accuracy_score

# %%
# 2. 載入資料
# 請將 raw_data.csv 放在當前工作目錄
# df = pd.read_csv('./raw_data/total.csv')
# df = pd.read_csv('./raw_data/age_below_66.csv')
# df = pd.read_csv('./raw_data/age_between_66_73.csv')
# df = pd.read_csv('./raw_data/age_over_73.csv')
# df = pd.read_csv('./raw_data/female_data.csv')
df = pd.read_csv('./raw_data/male_data.csv')

# Smote

#### 3. 區分連續與離散欄位

### 3. 建前處理器

# Set exp para

In [379]:
# ================== 0) 清欄名、切 X/y ==================
import numpy as np, pandas as pd
df = df.rename(columns=lambda c: c.strip()).rename(columns={'Hba1C':'HbA1c'})
TARGET = 'Second_Stroke'
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

# ================== 1) 欄位分組 ==================
bin_cols = ['sex','tPA(0/1)','EVT(0/1)','HTN(0/1)','DM(0/1)',
            'Dyslipidemia(0/1)','Af(0/1)','obstructive sleep apnea(0/1)','COVID-19(0/1)']
cat_cols = ['smoking(Y/N/Q)']     # 0/1/2 三類（名目）
ord_cols = ['MRS']
num_cont_cols  = ['age','LDL','cholesterol','TG','Cre','SGPT','HbA1c']
num_count_cols = ['HLOS','NIHSS']

feature_cols = list(X.columns)
keep = lambda cols: [c for c in cols if c in feature_cols]
bin_cols, cat_cols, ord_cols = keep(bin_cols), keep(cat_cols), keep(ord_cols)
num_cont_cols, num_count_cols = keep(num_cont_cols), keep(num_count_cols)

# ================== 2) 兩段式前處理 + SMOTENC ==================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, SVMSMOTE, ADASYN, SMOTENC

# Part-1: 先把連續特徵處理好、類別做 Ordinal（供 SMOTENC 用）
prep1 = ColumnTransformer([
    ('num_cont',  Pipeline([('imp', SimpleImputer(strategy='median')),
                            ('sc',  StandardScaler())]),               num_cont_cols),
    ('num_count', Pipeline([('imp', SimpleImputer(strategy='median')),
                            ('log', FunctionTransformer(np.log1p)),
                            ('sc',  StandardScaler())]),               num_count_cols),
    ('bin',       SimpleImputer(strategy='most_frequent'),             bin_cols),     # 保留 0/1 值
    ('ord',       SimpleImputer(strategy='median'),                    ord_cols),     # 先不縮放，當作類別
    ('cat_ord',   Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                            ('ord', OrdinalEncoder(handle_unknown='use_encoded_value',
                                                   unknown_value=-1))]),             cat_cols),
], remainder='drop')

# 計算 prep1 輸出中的各區段索引
def idx_range(start, length): return list(range(start, start+length))
n_cont, n_count, n_bin, n_ord, n_cat = map(len, [num_cont_cols, num_count_cols, bin_cols, ord_cols, cat_cols])

s0 = 0
s1 = s0 + n_cont
s2 = s1 + n_count
s3 = s2 + n_bin
s4 = s3 + n_ord
# s5 = s4 + n_cat  # 末端

idx_cont  = idx_range(s0, n_cont)
idx_count = idx_range(s1, n_count)
idx_bin   = idx_range(s2, n_bin)
idx_ord   = idx_range(s3, n_ord)
idx_cat   = idx_range(s4, n_cat)

categorical_features = idx_bin + idx_ord + idx_cat   # 告訴 SMOTENC 哪些是類別

# sampler = SMOTENC(categorical_features=categorical_features,
                #   sampling_strategy={1: NEG_PER_GROUP}, random_state=SEED, n_jobs=-1)

# sampler = SMOTE(
#     sampling_strategy={1: NEG_PER_GROUP},
#     random_state=SEED,
#     n_jobs=-1
# )

# sampler = BorderlineSMOTE(
#     kind='borderline-1',  # 可改 'borderline-2'
#     sampling_strategy={1: NEG_PER_GROUP},
#     random_state=SEED,
#     n_jobs=-1
# )

# sampler = SVMSMOTE(
#     sampling_strategy={1: NEG_PER_GROUP},
#     random_state=SEED,
#     n_jobs=-1
# )

sampler = ADASYN(
    sampling_strategy={1: NEG_PER_GROUP},
    random_state=SEED,
    n_jobs=-1
)

# Part-2: SMOTE 後再做 One-Hot / 縮放（以 ndarray 索引切片）
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

prep2 = ColumnTransformer([
    ('cont',   'passthrough', idx_cont),       # 已縮放
    ('count',  'passthrough', idx_count),      # 已log1p+縮放
    ('bin',    'passthrough', idx_bin),        # 0/1
    ('ord',    StandardScaler(), idx_ord),     # 這時才縮放有序分數
    ('cat',    ohe,            idx_cat),       # 將 Ordinal 的名目再一熱
], remainder='drop')

def make_pipeline(base_estimator):
    return Pipeline(steps=[
        ('prep1', prep1),
        ('smote', sampler),   # ← SMOTENC
        ('prep2', prep2),
        ('model', base_estimator),
    ])

# ================== 3) 10 組資料（固定 validation） ==================
rng = np.random.default_rng(SEED)

pos_idx = y.index[y==1].to_numpy()
neg_idx = y.index[y==0].to_numpy()

val_pos = rng.choice(pos_idx, size=N_VAL_POS, replace=False)
pos_train_pool = np.setdiff1d(pos_idx, val_pos)
val_neg = rng.choice(neg_idx, size=N_VAL_NEG, replace=False)
neg_train_pool = np.setdiff1d(neg_idx, val_neg)
val_idx = np.concatenate([val_pos, val_neg])
neg_groups = np.array_split(rng.permutation(neg_train_pool), N_NEG_GROUPS)

def get_fold_data(k: int):
    tr_idx = np.concatenate([pos_train_pool, neg_groups[k]])
    return (df.loc[tr_idx].drop(columns=[TARGET]),
            df.loc[tr_idx, TARGET].astype(int),
            df.loc[val_idx].drop(columns=[TARGET]),
            df.loc[val_idx, TARGET].astype(int))

# ================== 4) 模型們（含 XBC 與多種常見模型） ==================
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier

def build_models(SEED=42):
    models = {
        # 'SVM-Linear':  SVC(kernel='linear', C=1.0, gamma='scale',
        #                    probability=True, random_state=SEED),
        # 'SVM-Poly':   SVC(kernel='poly', degree=4, coef0=0.0, C=1.0, gamma='scale',
        #                    probability=True, random_state=SEED),
        'RF':       RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED),
        # 'AdaBoost': AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=SEED),
        # 'GBC':      GradientBoostingClassifier(random_state=SEED),

        # 'LR':       LogisticRegression(max_iter=2000, n_jobs=-1, random_state=SEED),
        # # 決策樹（與 SMOTENC 一起用，class_weight=None 即可）
        # 'DT': DecisionTreeClassifier(
        #     max_depth=None, min_samples_leaf=8, random_state=SEED
        # ),
    }

    # try:
    #     from xgboost import XGBClassifier
    #     models['XGBoost'] = XGBClassifier(
    #         n_estimators=600, max_depth=4, learning_rate=0.05,
    #         subsample=0.8, colsample_bytree=0.8,
    #         eval_metric='logloss', tree_method='hist', random_state=SEED, n_jobs=-1
    #     )
    # except Exception as e:
    #     print("ℹ️ XGBoost 不可用，略過。", e)

    # try:
    #     from lightgbm import LGBMClassifier
    #     models['LGBM'] = LGBMClassifier(
    #         n_estimators=1200, learning_rate=0.03,
    #         subsample=0.8, colsample_bytree=0.8,
    #         objective='binary', random_state=SEED, n_jobs=-1
    #     )
    # except Exception as e:
    #     print("ℹ️ LightGBM 不可用，略過。", e)

    

    return models

models = build_models(SEED)

In [380]:
from matplotlib.ticker import FormatStrFormatter
from xgboost import plot_importance
import matplotlib.pyplot as plt
import os

def build_feature_names_after_prep2(pipe, 
                                    num_cont_cols, num_count_cols, 
                                    bin_cols, ord_cols, cat_cols):
    """
    按照 prep2 = [('cont'),('count'),('bin'),('ord'),('cat')] 的輸出順序
    產生對齊的欄位名稱清單。
    """
    names = []
    # 1) 連續
    names += [f'num_cont__{c}' for c in num_cont_cols]
    # 2) 計數/分數
    names += [f'num_count__{c}' for c in num_count_cols]
    # 3) 二元
    names += [f'bin__{c}' for c in bin_cols]
    # 4) 有序
    names += [f'ord__{c}' for c in ord_cols]
    # 5) 名目（一熱後多欄）
    ohe = pipe.named_steps['prep2'].named_transformers_['cat']
    # 一個名目欄位對應一組 categories_
    cat_names = []
    for col, cats in zip(cat_cols, ohe.categories_):
        cat_names += [f'cat__{col}={v}' for v in cats]
    names += cat_names
    return names

def _strip_prefix(label: str) -> str:
    # 先吃掉 "bin__"、"num_count__" 這種；沒有時再吃掉第一個 "_"
    if "__" in label:
        return label.split("__", 1)[1]
    if "_" in label:
        return label.split("_", 1)[1]
    return label
def plot_rf_importance_with_names(pipe,
                                  num_cont_cols, num_count_cols,
                                  bin_cols, ord_cols, cat_cols,
                                  topk=20, title=None,
                                  save_path=None, show=False,
                                  decimals=2, annotate=True, fontsize=9,
                                  normalize=False):
    """從含 prep1→SMOTE→prep2→model 的 pipeline 讀出 RF 重要性並畫圖"""
    model = pipe.named_steps['model']
    if not hasattr(model, "feature_importances_"):
        raise ValueError("目前 model 沒有 feature_importances_，請確認是 RandomForest/ExtraTrees 等。")

    # 取得對齊的特徵名稱（你已經有同名函式，可重用）
    feat_names = build_feature_names_after_prep2(
        pipe, num_cont_cols, num_count_cols, bin_cols, ord_cols, cat_cols
    )

    importances = np.asarray(model.feature_importances_, dtype=float)
    if normalize and importances.sum() > 0:
        importances = importances / importances.sum()

    # 防呆：長度對不上就回報
    if len(importances) != len(feat_names):
        print(f"[WARN] len(importances)={len(importances)} != len(feature_names)={len(feat_names)}")

    # 取 topk（由大到小）
    idx = np.argsort(importances)[-topk:][::-1]
    vals = importances[idx]
    labels = [_strip_prefix(feat_names[i]) for i in idx]

    # 畫圖（手動控制 ticks，避免 FixedFormatter 錯）
    y = np.arange(len(idx))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(y, vals)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()  # 最大在最上方
    ax.set_xlabel("Feature Importance")
    ax.set_ylabel("Features")
    ax.set_title(title or "RandomForest Feature Importance")
    ax.xaxis.set_major_formatter(FormatStrFormatter(f'%.{decimals}f'))

    if annotate:
        xmin, xmax = ax.get_xlim(); dx = 0.01 * (xmax - xmin)
        for yi, v in zip(y, vals):
            ax.text(v + dx, yi, f"{(v*100 if normalize else v):.{decimals}f}" + ("%" if normalize else ""),
                    va="center", ha="left", fontsize=fontsize)

    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()

def plot_xgb_importance_with_names(pipe, 
                                   num_cont_cols, num_count_cols, 
                                   bin_cols, ord_cols, cat_cols,
                                   importance_type='weight', topk=20, title=None,
                                   save_path=None, show=False,
                                   decimals=2, annotate=True, fontsize=9):
    from matplotlib.ticker import FormatStrFormatter
    booster = pipe.named_steps['model'].get_booster()

    # 建立並指派特徵名稱（你原本的函式）
    feat_names = build_feature_names_after_prep2(
        pipe, num_cont_cols, num_count_cols, bin_cols, ord_cols, cat_cols
    )
    booster.feature_names = feat_names

    # 1) 不要內建的數值標籤
    ax = plot_importance(
        booster,
        importance_type=importance_type,
        max_num_features=topk,
        show_values=False,               # <<< 關掉第二排數字
    )

    # 2) 只保留第一個底線之後的名稱
    old = [t.get_text() for t in ax.get_yticklabels()]
    ax.set_yticklabels([_strip_prefix(t) for t in old])

    ax.set_xlabel("Feature Importance")
    ax.set_ylabel("Features")
    ax.set_title(title or f"XGBoost Feature Importance ({importance_type})")

    # 3) 自己加上到小數第2位的標籤
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    if annotate:
        xmin, xmax = ax.get_xlim()
        dx = 0.01 * (xmax - xmin)
        for p in ax.patches:
            w = p.get_width()
            y = p.get_y() + p.get_height()/2
            ax.text(w + dx, y, f'{w:.2f}', va='center', ha='left', fontsize=fontsize)

    import os, matplotlib.pyplot as plt
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close()


In [381]:
# ================== 5) 用 validation 找最佳閾值（以 F1 最大化）並評估 ==================
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support, f1_score

def find_best_threshold(y_true, prob, grid=np.linspace(0.05, 0.95, 181)):
    f1s = [f1_score(y_true, (prob >= t).astype(int), zero_division=0) for t in grid]
    i = int(np.argmax(f1s))
    return float(grid[i]), float(f1s[i])

def eval_from_score(y_true, score, name, store_thr=True):
    """給一個連續分數(score)，找 F1 最佳閾值並回傳 metrics 與閾值。"""
    t, _ = find_best_threshold(y_true, score)
    pred = (score >= t).astype(int)
    auc  = roc_auc_score(y_true, score)
    ap   = average_precision_score(y_true, score)
    acc  = accuracy_score(y_true, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
    row = {'model': name, 'thr': t, 'AUC': auc, 'ACC': acc, 'PREC': prec, 'REC': rec, 'F1': f1, 'F1*': f1}
    return row

records = []
SAVE_DIR_XG = "./figs/xgb_importance"  
SAVE_DIR_RF = "./figs/rf_importance"

# 先跑所有基模型，保存 proba / thr / AP
fold_proba = {}     # {model_name: proba vector}
fold_thr   = {}     # {model_name: best threshold on this fold}
fold_ap    = {}     # {model_name: AP on this fold}
for name, est in models.items():
    fold_proba[name] = []   # 每個 model 初始化一個空 list
    fold_thr[name]   = []
    fold_ap[name]    = []

rf_fold_importances  = []   # list of Series (index=feat, value=imp)
xgb_fold_importances = []   # list of Series (index=feat, value=imp, normalized)

XGB_IMPORT_TYPE = 'gain' 
for k in range(N_NEG_GROUPS):
    X_tr, y_tr, X_val, y_val = get_fold_data(k)

    rf_fig_dir = Path("./figs/rf_importance")
    rf_fig_dir.mkdir(parents=True, exist_ok=True)
    
    for name, est in models.items():
        pipe = make_pipeline(est)
        pipe.fit(X_tr, y_tr)

        if name in ('XGBoost'):
            booster = pipe.named_steps['model'].get_booster()
            feat_names = build_feature_names_after_prep2(
                pipe, num_cont_cols, num_count_cols, bin_cols, ord_cols, cat_cols
            )
            booster.feature_names = feat_names  # 讓 get_score 回傳正確特徵名
            imp_dict = booster.get_score(importance_type=XGB_IMPORT_TYPE)
            s = pd.Series(imp_dict, dtype=float).reindex(feat_names).fillna(0.0)
            s = s / (s.sum() + 1e-12)  # 每折歸一化
            s.index = [_strip_prefix(x) for x in s.index]  # 去掉前綴，和圖一致
            s.name = f"fold_{k:02d}"
            xgb_fold_importances.append(s)

            plot_xgb_importance_with_names(
                pipe, num_cont_cols, num_count_cols, bin_cols, ord_cols, cat_cols,
                importance_type=XGB_IMPORT_TYPE, topk=20,
                title=f'Fold {k} - XGB ({XGB_IMPORT_TYPE})',
                save_path=f"{SAVE_DIR_RF}/fold_{k:02d}_xgb_importance_{XGB_IMPORT_TYPE}.png",
                show=False
            )

        if name == 'RF':
            save_path = f"{SAVE_DIR_RF}/fold_{k:02d}_rf_importance.png"
            plot_rf_importance_with_names(
                pipe, num_cont_cols, num_count_cols, bin_cols, ord_cols, cat_cols,
                topk=20, title=f"Fold {k} - RF (imp)",
                save_path=save_path, show=False, decimals=2, annotate=True
                # normalize=True  # 若想顯示百分比就打開
            )
            
            # 1) 拿出對齊的特徵名稱
            feat_names = build_feature_names_after_prep2(
                pipe, num_cont_cols, num_count_cols, bin_cols, ord_cols, cat_cols
            )
            feat_names = [_strip_prefix(x) for x in feat_names]  # 去掉前綴，與圖一致

            # 2) 取出數值
            imp = np.asarray(pipe.named_steps['model'].feature_importances_, dtype=float)

            # 3) 存成 Series（index=特徵名）
            s = pd.Series(imp, index=feat_names, name=f"fold_{k:02d}")
            rf_fold_importances.append(s)

            # 4) 也順手存一份每折 CSV（可選）
            s.sort_values(ascending=False).to_csv(rf_fig_dir / f"rf_importance_fold_{k:02d}.csv")

        proba = pipe.predict_proba(X_val)[:, 1]
        t, best_f1 = find_best_threshold(y_val, proba)
        pred = (proba >= t).astype(int)

        auc  = roc_auc_score(y_val, proba)
        ap   = average_precision_score(y_val, proba)
        acc  = accuracy_score(y_val, pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_val, pred, average='binary', zero_division=0)

        # 存下來，等一下投票用
        fold_proba[name].append(proba)
        fold_thr[name].append(t)
        fold_ap[name].append(ap)

        records.append({'fold': k, 'model': name,
                        'AUC': auc, 'ACC': acc,
                        'PREC': prec, 'REC': rec, 'F1': f1})


    # ===== 這一折做三種投票 =====
    # base_models = list(fold_proba.keys())
    # if len(base_models) >= 2:
    #     import numpy as np

    #     # Soft Voting: 平均
    #     mat = np.column_stack([fold_proba[m] for m in base_models])
    #     score_soft_mean = mat.mean(axis=1)
    #     row = eval_from_score(y_val, score_soft_mean, name='ENS_soft(mean)')
    #     row['fold'] = k
    #     records.append(row)

    #     # Soft 
    # 
    # 
    # : 以 AP 加權的平均
    #     w = np.array([max(fold_ap[m], 1e-12) for m in base_models], dtype=float)
    #     w = w / w.sum()
    #     score_soft_wap = (mat * w).sum(axis=1)
    #     row = eval_from_score(y_val, score_soft_wap, name='ENS_soft(wAP)')
    #     row['fold'] = k
    #     records.append(row)

    #     # Hard Voting: 先各自用最佳閾值二值化，再多數決；投票比例當作分數
    #     bin_preds = np.column_stack([(fold_proba[m] >= fold_thr[m]).astype(int) for m in base_models])
    #     vote_frac = bin_preds.mean(axis=1)  # 0~1 之間的投票比例
    #     row = eval_from_score(y_val, vote_frac, name='ENS_hard(maj)')
    #     row['fold'] = k
    #     records.append(row)
    # else:
    #     print(f"[fold {k:02d}] 可用模型少於 2 個，跳過投票。")
fold_records = pd.DataFrame(records)
# ====== XGBoost：10 折平均重要度（gain），取前 10 名畫圖與存檔 ======
if len(xgb_fold_importances) > 0:
    imp_df_xgb = pd.concat(xgb_fold_importances, axis=1).fillna(0.0)   # (n_features, n_folds)
    mean_imp_xgb = imp_df_xgb.mean(axis=1).sort_values(ascending=False)
    mean_imp_xgb.name = f"mean_{XGB_IMPORT_TYPE}"

    imp_df_xgb.to_csv(f"{SAVE_DIR_XG}/xgb_importance_all_folds_{XGB_IMPORT_TYPE}.csv")
    mean_imp_xgb.to_csv(f"{SAVE_DIR_XG}/xgb_importance_mean_{XGB_IMPORT_TYPE}.csv")

    top10_xgb = mean_imp_xgb.nlargest(10)
    top10_xgb.to_csv(f"{SAVE_DIR_XG}/xgb_importance_mean_top10_{XGB_IMPORT_TYPE}.csv")

    # 畫圖（前 10 名）
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 6))
    vals = top10_xgb.values
    labels = top10_xgb.index.tolist()
    plt.barh(range(len(vals)), vals)
    plt.gca().invert_yaxis()
    plt.yticks(range(len(vals)), labels)
    plt.xlabel("Mean Importance (normalized)")
    plt.ylabel("Feature")
    plt.title(f"XGBoost Feature Importance (Top 10 mean across folds) [{XGB_IMPORT_TYPE}]")
    xmax = vals.max() if len(vals) else 0
    for i, v in enumerate(vals):
        plt.text(v + xmax*0.01, i, f"{v:.3f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR_XG}/xgb_importance_mean_top10_{XGB_IMPORT_TYPE}.png", dpi=300, bbox_inches="tight")
    plt.close()

if len(rf_fold_importances) > 0:
    imp_df = pd.concat(rf_fold_importances, axis=1).fillna(0.0)   # 每折 RF importance
    mean_imp = imp_df.mean(axis=1).sort_values(ascending=False)   # 各特徵平均重要度
    mean_imp.name = "mean_importance"

    # 存完整 CSV；另外也存前 10 名 CSV
    mean_imp.to_csv(rf_fig_dir / "rf_importance_mean.csv")
    imp_df.to_csv(rf_fig_dir / "rf_importance_all_folds.csv")
    topk = 10
    top10 = mean_imp.nlargest(topk)
    top10.to_csv(rf_fig_dir / "rf_importance_mean_top10.csv")

    # 只畫前 10 名的圖
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 6))
    vals = top10.values
    labels = top10.index.tolist()

    bars = plt.barh(range(len(vals)), vals)
    plt.gca().invert_yaxis()                     # 最大在最上
    plt.yticks(range(len(vals)), labels)
    plt.xlabel("Mean Importance")
    plt.ylabel("Feature")
    plt.title("Feature Importance (Top 10 mean across folds)")

    # 在每個 bar 末端標數值
    xmax = vals.max()
    for i, v in enumerate(vals):
        plt.text(v + xmax*0.01, i, f"{v:.3f}", va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(rf_fig_dir / "rf_importance_mean_top10.png", dpi=300, bbox_inches="tight")
    plt.close()

    print(f"✅ 已輸出 Top-10 平均重要度圖與 CSV 至：{rf_fig_dir.resolve()}")
res = pd.DataFrame(records)


/usr/local/lib/python3.8/site-packages/imblearn/over_sampling/_adasyn.py:156: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(
/usr/local/lib/python3.8/site-packages/imblearn/over_sampling/_adasyn.py:156: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(
/usr/local/lib/python3.8/site-packages/imblearn/over_sampling/_adasyn.py:156: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(
/usr/local/lib/python3.8/site-packages/imblearn/over_sampling/_adasyn.py:156: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can 

✅ 已輸出 Top-10 平均重要度圖與 CSV 至：/workspace/figs/rf_importance


In [382]:
# 計算 hard voting
def hard_vote_models(fold_proba: dict, fold_thr: dict, tie_break="avg_proba"):
    results = {}

    for m in fold_proba.keys():
        proba_list = fold_proba[m]  # list of arrays (n_folds, n_samples)
        thr_list   = fold_thr[m]

        assert len(proba_list) == len(thr_list), f"{m}: proba/thr 數量不一致"

        # 每個 fold 先轉成 binary 預測
        bin_preds = [(np.asarray(p) >= t).astype(int) for p, t in zip(proba_list, thr_list)]
        y_stack   = np.vstack(bin_preds)   # shape = (n_folds, n_samples)

        votes = y_stack.sum(axis=0)
        n_folds = y_stack.shape[0]

        # 多數決
        final = (votes > n_folds / 2).astype(int)

        # 平手處理
        if n_folds % 2 == 0:
            ties = (votes == n_folds // 2)
            if np.any(ties):
                if tie_break == "positive":
                    final[ties] = 1
                elif tie_break == "negative":
                    final[ties] = 0
                elif tie_break == "avg_proba":
                    mean_proba = np.mean(np.vstack([np.asarray(p) for p in proba_list]), axis=0)
                    mean_thr   = np.mean(thr_list)
                    final[ties] = (mean_proba[ties] >= mean_thr).astype(int)

        results[m] = final  # (n_samples,)

    return results

# 計算 hard voting
final_preds = hard_vote_models(fold_proba, fold_thr, tie_break="avg_proba")

for model_name, preds in final_preds.items():
    print(f"Model: {model_name}")
    print(preds)
    print("Length:", len(preds))
    print("-" * 40)

        
# 彙總投票結果 (hard voting) 

# 把 dict values 疊成 (n_models, n_samples)
pred_stack = np.vstack([preds for preds in final_preds.values()])   # (3, 27)

# 轉置成 (n_samples, n_models)
pred_matrix = pred_stack.T   # (27, 3)

print("Shape:", pred_matrix.shape)  # (27, 3)

# 對每筆資料做 hard voting
votes = pred_matrix.sum(axis=1)
n_models = pred_matrix.shape[1]

final_per_sample = (votes > n_models / 2).astype(int)

# 如果模型數是偶數，還要額外處理平手：
ties = (votes == n_models // 2)
if np.any(ties):
    final_per_sample[ties] = 1   # 或改成 0，看你 tie-break 想要怎麼設


from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score
)

print(final_per_sample)

#  Claculate ensemble metrics 

# final_preds: {model_name: np.ndarray (n_samples,)}，你已經有
# y_val:      np.ndarray (n_samples,)                # ground truth 標籤

# (1) 堆疊 -> (n_models, n_samples)
pred_stack   = np.vstack([np.asarray(p).astype(int) for p in final_preds.values()])
pred_matrix  = pred_stack.T           # (n_samples, n_models)

# (2) 投票
votes        = pred_matrix.sum(axis=1)
n_models     = pred_matrix.shape[1]

# soft score (for AUC/AP)
ensemble_score = votes / n_models
# hard voting result (for ACC/P/R/F1)
ensemble_pred  = (votes > n_models / 2).astype(int)

# tie-break（偶數模型數時）
if n_models % 2 == 0:
    ties = (votes == n_models // 2)
    if np.any(ties):
        ensemble_pred[ties] = 1   # or 0，看你的規則

# (3) 計算 ensemble metrics (跟 y_val 比！)
ens_auc  = roc_auc_score(y_val, ensemble_score)
ens_ap   = average_precision_score(y_val, ensemble_score)
ens_acc  = accuracy_score(y_val, ensemble_pred)
ens_prec = precision_score(y_val, ensemble_pred, zero_division=0)
ens_rec  = recall_score(y_val, ensemble_pred)
ens_f1   = f1_score(y_val, ensemble_pred)

# (4) 做成一筆新紀錄加進 res
ensemble_record = {
    'model': 'Ensemble-Hard',
    'AUC':   ens_auc,
    'ACC':   ens_acc,
    'PREC':  ens_prec,
    'REC':   ens_rec,
    'F1':    ens_f1,
}

res = pd.concat([res, pd.DataFrame([ensemble_record])], ignore_index=True)

Model: RF
[1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 1 1]
Length: 18
----------------------------------------
Shape: (18, 1)
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [383]:
# 計算 soft voting
# 把三個模型（SVM、RF、XGBoost）各自 10 folds 的 機率先做「模型內 soft voting」（平均機率），
# 再做「模型間 soft voting」（平均或加權平均），
# 最後輸出 每筆 validation data（27 筆） 的 ensemble 結果。

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score
)

# ========= 核心：兩層 soft voting with options =========
from sklearn.metrics import f1_score

def soft_vote_ensemble(
    fold_proba,
    fold_thr=None,
    fold_ap=None,
    weights="uniform",
    decision="mean_thr",
    y_true=None,
):
    models = list(fold_proba.keys())

    # 1) 模型內 soft vote：各模型 mean proba
    per_model_mean_proba, per_model_mean_thr = {}, {}
    for m in models:
        P = np.vstack([np.asarray(p, dtype=float) for p in fold_proba[m]])   # (n_folds, n_samples)
        per_model_mean_proba[m] = P.mean(axis=0)                              # (n_samples,)
        if fold_thr is not None and m in fold_thr and len(fold_thr[m]) > 0:
            per_model_mean_thr[m] = float(np.mean(fold_thr[m]))
        else:
            per_model_mean_thr[m] = np.nan

    # 2) 權重
    if weights == "ap" and fold_ap is not None:
        raw_w = np.array([np.mean(fold_ap[m]) for m in models], dtype=float)
        w = raw_w if (np.isfinite(raw_w).all() and np.any(raw_w > 0)) else np.ones_like(raw_w)
    else:
        w = np.ones(len(models), dtype=float)
    w = w / w.sum()
    model_weights = {m: float(w_i) for m, w_i in zip(models, w)}

    # 3) 模型間 soft vote：加權平均
    stack = np.vstack([per_model_mean_proba[m] for m in models])  # (n_models, n_samples)
    ensemble_score = np.average(stack, axis=0, weights=w)         # (n_samples,)

    # 4) 門檻策略
    if decision == "mean_thr":
        thr_vals = np.array([per_model_mean_thr[m] for m in models], dtype=float)
        if np.isnan(thr_vals).all():
            ensemble_thr = 0.5
        else:
            if np.any(np.isnan(thr_vals)):
                fill = np.nanmean(thr_vals) if np.isfinite(np.nanmean(thr_vals)) else 0.5
                thr_vals = np.where(np.isnan(thr_vals), fill, thr_vals)
            ensemble_thr = float(np.average(thr_vals, weights=w))
    elif decision == "fixed_0.5":
        ensemble_thr = 0.5
    elif decision == "tune_f1":
        if y_true is None:
            raise ValueError("decision='tune_f1' 需要提供 y_true")
        grid = np.linspace(0, 1, 1001)
        f1s = [f1_score(y_true, (ensemble_score >= t).astype(int), zero_division=0) for t in grid]
        ensemble_thr = float(grid[int(np.argmax(f1s))])
    else:
        raise ValueError("decision 只能是 'mean_thr' | 'fixed_0.5' | 'tune_f1'")

    ensemble_pred = (ensemble_score >= ensemble_thr).astype(int)

    return {
        "model_weights": model_weights,
        "per_model_mean_thr": per_model_mean_thr,
        "ensemble_score": ensemble_score,
        "ensemble_thr": ensemble_thr,
        "ensemble_pred": ensemble_pred,
    }

out = soft_vote_ensemble(
    fold_proba=fold_proba,
    fold_thr=fold_thr,
    fold_ap=fold_ap,       # 如果要用 AP 加權，記得 weights="ap"
    weights="ap",     # 或 "ap"
    decision="tune_f1",
    y_true=y_val
)

# 用 score 算 AUC/AP；用 pred 算 ACC/P/R/F1
score = out["ensemble_score"]
pred  = out["ensemble_pred"]
thr   = out["ensemble_thr"]

ens_auc  = roc_auc_score(y_val, score)
ens_ap   = average_precision_score(y_val, score)
ens_acc  = accuracy_score(y_val, pred)
ens_prec = precision_score(y_val, pred, zero_division=0)
ens_rec  = recall_score(y_val, pred)
ens_f1   = f1_score(y_val, pred)

model_name = "Ensemble-Soft"
ens_record = {
    "model": model_name,
    "AUC": ens_auc, "AP": ens_ap, "ACC": ens_acc,
    "PREC": ens_prec, "REC": ens_rec, "F1": ens_f1, "thr": thr
}

# 印出結果表
ens_cmp = (pd.DataFrame([ens_record])
             .set_index("model")
             .round(4))
print("\n=== Ensemble-Soft (tune_f1) ===")
print(ens_cmp[["AUC","AP","ACC","PREC","REC","F1","thr"]])

# 加進 res 再算 summary
res = pd.concat([res, pd.DataFrame([ens_record])], ignore_index=True)





=== Ensemble-Soft (tune_f1) ===
                  AUC      AP     ACC    PREC     REC      F1    thr
model                                                               
Ensemble-Soft  0.6049  0.6746  0.6111  0.5714  0.8889  0.6957  0.183


In [384]:
from pathlib import Path

# 存每個 fold 的完整資料
Path("./result").mkdir(parents=True, exist_ok=True)
fold_records.round(2).to_csv(f"./result/folds_seed{SEED}.csv", index=False)

summary = (res.groupby('model')
             .agg({'AUC':['mean','std'], 'ACC':['mean','std'],
                   'PREC':['mean','std'],'REC':['mean','std'],'F1':['mean','std']})
             .round(4))
print(f'SEED = {SEED}')
print(summary)

summary.to_csv(f"./result/summary_seed{SEED}.csv")

# 想看每折的投票最佳閾值：
# res[res['model'].str.startswith('ENS_')].pivot_table(index='fold', columns='model', values='thr')


SEED = 95
                  AUC            ACC            PREC             REC          \
                 mean    std    mean     std    mean     std    mean     std   
model                                                                          
Ensemble-Hard  0.6111    NaN  0.6111     NaN  0.5625     NaN  1.0000     NaN   
Ensemble-Soft  0.6049    NaN  0.6111     NaN  0.5714     NaN  0.8889     NaN   
RF             0.5926  0.109  0.6056  0.0996  0.5735  0.0779  0.9556  0.0574   

                   F1          
                 mean     std  
model                          
Ensemble-Hard  0.7200     NaN  
Ensemble-Soft  0.6957     NaN  
RF             0.7118  0.0478  
